# Data Visualization

## Common Settings

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    from IPython.core.getipython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.run_line_magic('load_ext', 'autoreload')
        ipython.run_line_magic('autoreload', '2')
except ImportError:
    pass

from src import (
    DEFAULT_CONFIG,
    read_excel_data,
    set_pub_style,
    to_long,
    plot_fnorm,
    plot_pareto_scatter,
    plot_combined_scalability_fnorm,
    plot_network_robustness,
    merge_pdfs_in_order,
)

BENCHMARK_INPUT_XLSX = ROOT / 'data' / '4x20.xlsx'
BENCHMARK_SHEET_NAME = 'Sheet1'
NETWORK_INPUT_XLSX = ROOT / 'data' / '4x50.xlsx'
NETWORK_SHEET_NAME = 'Sheet1'
SCALABILITY_QPU_INPUT_XLSX = ROOT / 'data' / 'scalability_qpu.xlsx'
SCALABILITY_QPU_SHEET_NAME = 'Sheet1'
SCALABILITY_QUBITS_INPUT_XLSX = ROOT / 'data' / 'scalability_qubits.xlsx'
SCALABILITY_QUBITS_SHEET_NAME = 'Sheet1'
OUTPUT_DIR = ROOT / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

set_pub_style()

print('ROOT =', ROOT)
print('BENCHMARK_INPUT_XLSX =', BENCHMARK_INPUT_XLSX)
print('BENCHMARK_SHEET_NAME =', BENCHMARK_SHEET_NAME)
print('NETWORK_INPUT_XLSX =', NETWORK_INPUT_XLSX)
print('NETWORK_SHEET_NAME =', NETWORK_SHEET_NAME)
print('SCALABILITY_QPU_INPUT_XLSX =', SCALABILITY_QPU_INPUT_XLSX)
print('SCALABILITY_QPU_SHEET_NAME =', SCALABILITY_QPU_SHEET_NAME)
print('SCALABILITY_QUBITS_INPUT_XLSX =', SCALABILITY_QUBITS_INPUT_XLSX)
print('SCALABILITY_QUBITS_SHEET_NAME =', SCALABILITY_QUBITS_SHEET_NAME)
print('OUTPUT_DIR =', OUTPUT_DIR)


## Data Analysis

In [ ]:
from src import analyze_workbook
import pandas as pd

DATA_ANALYSIS_JOBS = [
    (BENCHMARK_INPUT_XLSX, OUTPUT_DIR / '4x20_analysis.xlsx'),
    (SCALABILITY_QPU_INPUT_XLSX, OUTPUT_DIR / 'scalability_qpu_analysis.xlsx'),
    (SCALABILITY_QUBITS_INPUT_XLSX, OUTPUT_DIR / 'scalability_qubits_analysis.xlsx'),
]

analysis_results = []

for input_xlsx, output_xlsx in DATA_ANALYSIS_JOBS:
    if not input_xlsx.exists():
        raise FileNotFoundError(f'Input workbook not found: {input_xlsx}')

    analyzed_sheets = analyze_workbook(
        input_xlsx=input_xlsx,
        output_xlsx=output_xlsx,
        bench_order=DEFAULT_CONFIG.bench_order,
    )
    analysis_results.append(
        {
            'input_xlsx': input_xlsx,
            'output_xlsx': output_xlsx,
            'sheets': ', '.join(analyzed_sheets.keys()),
        }
    )

df_analysis_results = pd.DataFrame(analysis_results)
display(df_analysis_results)


## Benchmark Details

In [ ]:
df_wide = read_excel_data(BENCHMARK_INPUT_XLSX, BENCHMARK_SHEET_NAME)
df_long = to_long(df_wide)

meta_fnorm = plot_fnorm(
    df_long,
    order=DEFAULT_CONFIG.bench_order,
    output_prefix=OUTPUT_DIR / 'benchmarks_fnorm',
    cfg=DEFAULT_CONFIG,
)

meta_pareto = plot_pareto_scatter(
    df_long,
    order=DEFAULT_CONFIG.bench_order,
    output_prefix=OUTPUT_DIR / 'benchmarks_pareto',
    cfg=DEFAULT_CONFIG,
)

# display(meta_fnorm)
# display(meta_pareto)


## Scalability

In [ ]:
df_scalability_qpu = read_excel_data(SCALABILITY_QPU_INPUT_XLSX, SCALABILITY_QPU_SHEET_NAME)
df_long_scalability_qpu = to_long(df_scalability_qpu)

df_scalability_qubit = read_excel_data(SCALABILITY_QUBITS_INPUT_XLSX, SCALABILITY_QUBITS_SHEET_NAME)
df_long_scalability_qubit = to_long(df_scalability_qubit)

meta_scalability_fnorm = plot_combined_scalability_fnorm(
    df_long_scalability_qpu=df_long_scalability_qpu,
    df_long_qubits=df_long_scalability_qubit,
    output_prefix=OUTPUT_DIR / 'scalability',
    cfg=DEFAULT_CONFIG,
)

# display(meta_scalability_fnorm)


## Network Topology Robustness

In [ ]:
meta_topology = plot_network_robustness(
    input_path=NETWORK_INPUT_XLSX,
    output_prefix=OUTPUT_DIR / 'networks',
    config=DEFAULT_CONFIG,
    sheet_name=NETWORK_SHEET_NAME,
    show_in_notebook=True,
)

# meta_topology.head()


## Merge PDF

In [ ]:
merged_pdf, merged_list = merge_pdfs_in_order(
    [
        OUTPUT_DIR / 'overview.pdf',
        OUTPUT_DIR / 'benchmarks_fnorm.pdf',
        OUTPUT_DIR / 'benchmarks_pareto.pdf',
        OUTPUT_DIR / 'scalability.pdf',
        OUTPUT_DIR / 'networks.pdf',
    ],
    output_pdf=OUTPUT_DIR / 'figs.pdf',
)

print('Merged PDF:', merged_pdf)
print('Included files:')
for pdf in merged_list:
    print('-', pdf)
